# Manual Tracing: Add Custom Spans to Any Application

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/quickstart/manual-tracing)

Follow a Ridgeline support request from document retrieval to the generated answer. Use the Tracing dashboard to inspect the span tree.

## Install

Use a fresh Python 3.12 kernel. The key cell uses existing environment variables or prompts without displaying your input.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "fi-instrumentation-otel==1.5.1", "traceAI-openai==0.1.16", "openai==1.109.1"],
    check=True,
);

In [ ]:
import os
from getpass import getpass

for name in ['FI_API_KEY', 'FI_SECRET_KEY', 'OPENAI_API_KEY']:
    if not os.environ.get(name):
        value = getpass(f"{name}: ").strip()
        if not value:
            raise ValueError(f"{name} is required")
        os.environ[name] = value
        del value

# For a self-hosted deployment, set FI_TRACES_URL to your HTTP collector.
os.environ.setdefault("FI_TRACES_URL", "https://api.futureagi.com");

## Auto-trace every OpenAI call

Open **Tracing → ridgeline-support** after the call to see its model, input and output.

In [ ]:
from fi_instrumentation import register
from fi_instrumentation.fi_types import ProjectType
from traceai_openai import OpenAIInstrumentor
from openai import OpenAI

# 1. Register the tracer provider
trace_provider = register(
    project_type=ProjectType.OBSERVE,
    project_name="ridgeline-support",
    verbose=False,
)

# 2. Patch the OpenAI client
OpenAIInstrumentor().instrument(tracer_provider=trace_provider)

# All subsequent OpenAI calls are now traced automatically
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "How many days are in a 30-day return window?"}],
)
print(response.choices[0].message.content)

## Add a custom span for non-LLM steps

In [ ]:
from fi_instrumentation import FITracer

# Get a tracer scoped to this module. FITracer applies context attributes to custom spans
tracer = FITracer(trace_provider.get_tracer(__name__))

def retrieve_context(query: str) -> list[str]:
    with tracer.start_as_current_span("retrieve-context") as span:
        span.set_attribute("retrieval.query", query)

        # Simulate a vector DB lookup
        docs = ["Ridgeline accepts unused gear within 30 days.", "Orders over $50 ship free."]

        span.set_attribute("retrieval.doc_count", len(docs))
        return docs


def answer_with_context(query: str) -> str:
    # Parent span groups retrieval + LLM into one trace
    with tracer.start_as_current_span("answer-with-context") as span:
        docs = retrieve_context(query)
        context = "\n".join(docs)

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": f"Answer using only this context:\n{context}"},
                {"role": "user", "content": query},
            ],
        )
        return response.choices[0].message.content


print(answer_with_context("When can I return unused gear?"))

## Attach user ID, session ID, and metadata

Use these synthetic identifiers to find the request in the Tracing dashboard.

In [ ]:
from fi_instrumentation import using_user, using_session, using_metadata

user_id = "user-abc123"
session_id = "session-xyz789"
metadata = {"environment": "production", "app_version": "2.1.0"}

with using_user(user_id), using_session(session_id), using_metadata(metadata):
    # Both the retrieval span and the OpenAI span get user.id, session.id, and metadata
    result = answer_with_context("How many days are in a 30-day return window?")
    print(result)

## Tag spans for filtering and alerting

Filter `tag.tags` for `rag-pipeline` to find this request.

In [ ]:
from fi_instrumentation import using_tags

# Tag all traces from this run as production + rag-pipeline
with using_tags(["production", "rag-pipeline", "v2"]):
    result = answer_with_context("When does an order qualify for free shipping?")
    print(result)

## Nest spans for complex multi-step pipelines

In [ ]:
def run_rag_pipeline(user_query: str, user_id: str, session_id: str) -> str:
    with using_user(user_id), using_session(session_id), using_tags(["rag-pipeline"]):
        with tracer.start_as_current_span("rag-pipeline") as pipeline_span:
            pipeline_span.set_attribute("pipeline.query", user_query)

            # Child span 1: retrieval
            with tracer.start_as_current_span("retrieve") as retrieval_span:
                docs = retrieve_context(user_query)
                retrieval_span.set_attribute("retrieval.doc_count", len(docs))

            # Child span 2: LLM call (auto-instrumented, just call it)
            context_text = "\n".join(docs)
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": f"Answer using:\n{context_text}"},
                    {"role": "user", "content": user_query},
                ],
            )
            answer = response.choices[0].message.content
            pipeline_span.set_attribute("pipeline.answer_length", len(answer))
            return answer


result = run_rag_pipeline(
    user_query="What is the return window for unused gear?",
    user_id="user-abc123",
    session_id="session-xyz789",
)
print(result)

## Log prompt template details with using_prompt_template

The context records a template reference on the span; it does not create a Prompt Workbench template.

In [ ]:
from fi_instrumentation import using_prompt_template

# These values should match a prompt created in your Prompt Workbench
with using_prompt_template(
    template="support-response",
    label="production",
    version="v2",
    variables={"question": "What is the return policy?"},
):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Ridgeline accepts unused items within 30 days."},
            {"role": "user", "content": "What is the return policy?"},
        ],
    )
    print(response.choices[0].message.content)

## Use decorators for agent and tool spans

In [ ]:
# FITracer was imported in Step 2, reuse it here
# tracer = FITracer(trace_provider.get_tracer(__name__))


@tracer.agent(name="support_agent")
def support_agent(question: str) -> str:
    """Top-level agent that orchestrates retrieval and generation."""
    docs = search_docs(question)
    return generate_answer(question, docs)


@tracer.tool(name="search_docs", description="Search the product documentation")
def search_docs(query: str) -> list[str]:
    return ["30-day return policy for unused items.", "Free shipping on orders over $50."]


@tracer.chain(name="generate_answer")
def generate_answer(question: str, docs: list[str]) -> str:
    context = "\n".join(docs)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Answer using:\n{context}"},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content


result = support_agent("What is the return policy?")
print(result)

trace_provider.force_flush()

In the `ridgeline-support` project, open the `support_agent` trace. Its children are the retrieval tool and the answer-generation chain; the OpenAI span is inside the chain.